### Circuits: Bronze to Silver
Clean and transform raw circuits data from `formula1_incr.bronze.circuits` into `formula1_incr.silver.circuits`.

#### Setup
- `01.environment-config` → loads catalog name, bronze/silver schema names
- `03.silver_helpers` → loads the `write_to_silver()` function we use to save data

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver_helpers

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import functions as f

In [0]:
dbutils.widgets.text('p_batch_id','')
v_batch_id= dbutils.widgets.get('p_batch_id')

In [0]:

bronze_table = f'{catalog_name}.{bronze_schema}.circuits'
silver_table = f'{catalog_name}.{silver_schema}.circuits'

#### Read Bronze
- Read raw data from the bronze table filtered by `batch_id`

##### Note
We use `spark.read.table()` to read from an existing Delta table.

In [0]:
circuits_df = (spark.read.table(bronze_table).filter(col('batch_id')==v_batch_id))
# or circuits_df = spark.table(bronze_table) best is to use read.table as it is more explicit

#### Drop Columns
- Remove `url` column — not needed for analysis

In [0]:
from pyspark.sql.functions import *
circuits_selected_df = circuits_df.select(
    col('circuitID'),
    col('circuitName'),
    col('lat'),
    col('long'),
    col('locality'),
    col('country'),
    col('ingestion_timestamp'),
    col('source_file'),
    col('batch_id')
)


#### Select Columns
- Keep only columns needed for analysis, drop everything else

#### Rename Columns
- Convert camelCase to snake_case for consistency

##### Note
`withColumnsRenamed()` renames multiple columns at once.

In [0]:
circuits_renamed_df = (circuits_selected_df
 .withColumnsRenamed({
   'circuitID':'circuit_id',                                          
   'circuitName':'circuit_name',                                          
   'lat':'latitude',
   'long':'longitude'})) # this uses withcolumn(S)

#### Remove Nulls
- Drop rows where `circuit_id` is null

In [0]:
display(circuits_renamed_df)


In [0]:
circuits_validate_df = circuits_renamed_df.filter(
    col('circuit_id').isNotNull() 
)

In [0]:
display(circuits_validate_df)  

#### Remove Duplicates
- Keep one row per circuit using `dropDuplicates()`

In [0]:
circuits_distinct_df = circuits_validate_df.distinct()

In [0]:
circuits_distinct_df = circuits_validate_df.dropDuplicates(["circuit_id"])
display(circuits_distinct_df)

In [0]:
circuits_distinct_df.createOrReplaceTempView("circuits_distinct_df")
spark.sql("select count(*) from circuits_distinct_df").display()

#### Title Case
- Apply `initcap()` to `circuit_name` and `locality` so they look clean

In [0]:
from pyspark.sql.functions import initcap
circuits_final_df =(circuits_distinct_df
 .withColumn('circuit_name', initcap(col('circuit_name')))
 .withColumn('locality', initcap(col('locality'))))
display(circuits_final_df)



#### Write to Silver
- If the silver table doesn't exist yet, it creates it from scratch
- If it already exists, it merges new/updated rows using `write_to_silver()` (insert new, update changed)

In [0]:
"""circuits_final_df = (
circuits_final_df
 .withColumn('created_timestamp', current_timestamp())
 .withColumn('updated_timestamp', current_timestamp()))"""

In [0]:
""" if not spark.catalog.tableExists(silver_table):
 (
  circuits_final_df
  .write
  .mode("overwrite")
  .format("delta")
  .saveAsTable(f'{catalog_name}.{silver_schema}.circuits')
 )
else:
 from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, silver_table)
(
   delta_table.alias('t')
   .merge(
       circuits_final_df.alias('s'),
       't.circuit_id = s.circuit_id'
   )
   .whenMatchedUpdate
   (
       condition = 's.batch_id >= t.batch_id',
       set =
    {
        'circuit_name': 's.circuit_name',
        'latitude': 's.latitude',
        'longitude': 's.longitude',
        'locality': 's.locality',
        'country': 's.country',
        'ingestion_timestamp': 's.ingestion_timestamp',
        'source_file': 's.source_file',
        'batch_id' : 's.batch_id',
        'updated_timestamp': 's.updated_timestamp'
    }
    )
   .whenNotMatchedInsertAll()
   .execute()
) """

In [0]:
write_to_silver(
    input_df = circuits_final_df,
    target_table = silver_table,
    merge_condition = "t.circuit_id = s.circuit_id",
    columns_to_update = [
        'circuit_name', 
        'latitude', 
        'longitude', 
        'locality', 
        'country', 
        'ingestion_timestamp', 
        'source_file',
        'batch_id']
    
)

In [0]:
spark.read.table(silver_schema).display()